# GLips Lippenlesen mit Deep Learning

## Datenimport
Der Datensatz besteht aus Videos aus dem Hessischen Parlament wobei 500 Verschiedene Wörter enthalten sind. Die Videos sind bereits auf die Lippen gecropt und in 25 Frames pro Sekunde umgewandelt worden. Der Folder ist in Wörter unterteilt, die jeweils train test und validation Ordner enthalten.

In [2]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.io as io

class GLipsFullClipDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None, num_frames=25):
        self.root_dir = root_dir
        self.split = split
        self.transform = transform
        # target number of frames per sample (videos will be truncated or padded)
        self.num_frames = num_frames
        self.samples = []

        # collect class folders
        self.classes = sorted([d for d in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        # For each class folder, look for the requested split folder. If not found, try common alternatives.
        for cls_name in self.classes:
            split_folder = os.path.join(root_dir, cls_name, split)
            if not os.path.exists(split_folder):
                for alt in ['train', 'validation', 'val', 'test']:
                    alt_folder = os.path.join(root_dir, cls_name, alt)
                    if os.path.exists(alt_folder):
                        split_folder = alt_folder
                        break

            if os.path.exists(split_folder):
                for file in os.listdir(split_folder):
                    if file.endswith('.mp4'):
                        self.samples.append((os.path.join(split_folder, file), self.class_to_idx[cls_name]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        video_path, label = self.samples[idx]
        # Load entire video: try multiple backends for compatibility
        # Desired tensor shape after loading: (T, C, H, W)
        video = None

        # 1) torchvision (if available)
        if hasattr(io, 'read_video'):
            try:
                video, _, _ = io.read_video(video_path, pts_unit='sec', output_format='TCHW')
            except Exception:
                video = None

        # 2) imageio (ffmpeg) fallback
        if video is None:
            try:
                import imageio.v2 as imageio
            except Exception:
                try:
                    import imageio
                except Exception:
                    imageio = None

            if imageio is not None:
                frames = []
                try:
                    reader = imageio.get_reader(video_path, 'ffmpeg')
                    for frame in reader:
                        frames.append(frame)
                    reader.close()
                    import numpy as np
                    video_np = np.stack(frames)  # (T, H, W, C)
                    video = torch.from_numpy(video_np).permute(0, 3, 1, 2)  # (T, C, H, W)
                except Exception:
                    video = None

        # 3) OpenCV fallback
        if video is None:
            try:
                import cv2
                import numpy as np
                cap = cv2.VideoCapture(video_path)
                frames = []
                while True:
                    ret, frame = cap.read()
                    if not ret:
                        break
                    # convert BGR->RGB
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frames.append(frame)
                cap.release()
                if len(frames) == 0:
                    raise RuntimeError(f"No frames read from {video_path}")
                video_np = np.stack(frames)
                video = torch.from_numpy(video_np).permute(0, 3, 1, 2)
            except Exception as e:
                raise RuntimeError(
                    "Could not read video with torchvision, imageio or opencv. "
                    f"Install one of these backends. Original error: {e}"
                )

        # Convert to float in range [0,1]
        video = video.float() / 255.0

        # video currently: (T, C, H, W)
        T = video.size(0)

        # 1) If video longer than target, sample uniformly to reduce to num_frames
        if T > self.num_frames:
            import numpy as np
            indices = np.linspace(0, T - 1, num=self.num_frames).astype(int)
            video = video[indices]

        # 2) If video shorter than target, pad by repeating last frame
        elif T < self.num_frames:
            pad_count = self.num_frames - T
            last = video[-1:].repeat(pad_count, 1, 1, 1)
            video = torch.cat([video, last], dim=0)

        # Apply optional transform (expects tensor shape (T,C,H,W) or will handle accordingly)
        if self.transform:
            video = self.transform(video)

        # Returns (C, T, H, W) and label
        return video.permute(1, 0, 2, 3), label

In [3]:
# Create train and validation datasets/loaders (split folder names are tried with fallbacks inside the dataset)
train_dataset = GLipsFullClipDataset(root_dir='./GLips/lipread_files/', split='train')
val_dataset = GLipsFullClipDataset(root_dir='./GLips/lipread_files/', split='validation')

_pin = torch.cuda.is_available()
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0, pin_memory=_pin)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0, pin_memory=_pin)

## Modell
in README.md können sie sich die Modellarchitektur anschauen. Es handelt sich um ein 3D-CNN mit mehreren Convolutional und Pooling Schichten, gefolgt von einem Transformer und einem Neuralen Netzwerk. Das Modell ist darauf ausgelegt, die zeitlichen und räumlichen Merkmale der Lippenbewegungen zu erfassen.

### 3D CNN + ResNet
Mit dem 3DCNN werden die räumlichen und zeitlichen Merkmale der Lippenbewegungen extrahiert. Die Convolutional Schichten erfassen lokale und kleine zeitliche Muster. Der ResNet-Block ermöglicht es, detailierte Merkmale der Lippenbewegungen zu erfassen, indem er die Informationen über mehrere Schichten hinweg weitergibt. Dies hilft, das Problem des Vanishing Gradient zu vermeiden und ermöglicht es dem Modell, tiefere Netzwerke zu trainieren.

In [4]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


class Frontend3D(nn.Module):
    def __init__(self):
        super(Frontend3D, self).__init__()
        self.layers = nn.Sequential(
            nn.Conv3d(3, 64, kernel_size=(5, 7, 7), stride=(1, 2, 2), padding=(2, 3, 3), bias=False),
            nn.BatchNorm3d(64),
            nn.ReLU(True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1))
        )

    def forward(self, x):
        # x: (B, 3, T, H, W) -> (B, 64, T, H', W')
        return self.layers(x)


class ResNet2DBackend(nn.Module):
    def __init__(self):
        super(ResNet2DBackend, self).__init__()
        resnet = models.resnet18(weights=None)
        self.layers = nn.Sequential(*list(resnet.children())[4:-2])

    def forward(self, x):
        # x: (B*T, 64, H', W') -> (B*T, 512, H'', W'')
        return self.layers(x)


class CNN3D(nn.Module):
    def __init__(self):
        super(CNN3D, self).__init__()
        self.frontend = Frontend3D()
        self.resnet = ResNet2DBackend()

    def forward(self, x):
        # x: (B, 3, T, H, W)
        B, C, T, H, W = x.size()

        x = self.frontend(x)                          # (B, 64, T, H', W')
        x = x.transpose(1, 2).contiguous()            # (B, T, 64, H', W')
        x = x.view(-1, 64, x.size(3), x.size(4))      # (B*T, 64, H', W')
        x = self.resnet(x)                             # (B*T, 512, H'', W'')
        x = x.view(B, T, 512, x.size(2), x.size(3))   # (B, T, 512, H'', W'')

        return x  # Output: (B, T, 512, H'', W'')

## GRU
Der GRU-Teil des Modells erfasst die zeitlichen Abhängigkeiten in den extrahierten Merkmalen der Lippenbewegungen. Er ist bidirektional, sodass jeder Frame sowohl vergangene als auch zukünftige Kontextinformationen nutzen kann. Im Vergleich zum Transformer ist der GRU deutlich schneller und speichereffizienter.

In [5]:
import torch
import torch.nn as nn

class LipreadingGRU(nn.Module):
    def __init__(self, input_size=512, hidden_size=512, num_layers=2, dropout=0.1):
        super(LipreadingGRU, self).__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        self.proj = nn.Linear(hidden_size * 2, 512)  # *2 for bidirectional

    def forward(self, x):
        x, _ = self.gru(x)   # (B, T, hidden*2)
        return self.proj(x)   # (B, T, 512)

## Orchestrator und Linear
Hier werden die verschiedenen Komponenten des Modells zusammengeführt. Der Orchestrator verbindet das 3D-CNN mit dem Transformer, um die extrahierten Merkmale der Lippenbewegungen zu verarbeiten. Am Ende des Orchestrators befindet sich eine lineare Schicht, die die Ausgabe des Transformers in die gewünschte Anzahl von Klassen (Wörtern) umwandelt, um die Vorhersage zu ermöglichen.

In [6]:
class GLipsModel(nn.Module):
    def __init__(self, num_classes=500):
        super(GLipsModel, self).__init__()
        self.cnn = CNN3D()
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        #self.gru = LipreadingGRU(input_size=512, hidden_size=512)
        self.classifier = nn.Linear(512, num_classes)

    def forward(self, x):
        # 1. 3D-CNN + ResNet: (B, 3, T, H, W) -> (B, T, 512, H'', W'')
        x = self.cnn(x)

        # 2. Avg pool over spatial dims: (B, T, 512, H'', W'') -> (B, T, 512)
        B, T, C, H, W = x.size()
        x = x.view(B * T, C, H, W)
        x = self.avgpool(x)          # (B*T, 512, 1, 1)
        x = x.view(B, T, 512)

        # 3. GRU: (B, T, 512) -> (B, T, 512)
        #x = self.gru(x)

        # 4. Mean pooling over time: (B, 512)
        x = torch.mean(x, dim=1)

        # 5. Classification: (B, 500)
        return self.classifier(x)

## Training

In [7]:
from tqdm.auto import tqdm

Model = GLipsModel()

optimizer = torch.optim.Adam(Model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
Model.to(device)

# Mixed precision scaler (no-op on CPU)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

num_epochs = 100

best_val_acc = 0.0
save_dir = './checkpoints'
os.makedirs(save_dir, exist_ok=True)

epoch_bar = tqdm(range(num_epochs), desc="Epochs")

for epoch in epoch_bar:
    # ── Training ──────────────────────────────────────────────────────────────
    Model.train()
    running_loss = 0.0
    num_samples = 0

    train_bar = tqdm(train_loader, desc=f"Train {epoch+1}/{num_epochs}", leave=False)
    for data, target in train_bar:
        data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            output = Model(data)
            loss = criterion(output, target)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = data.size(0)
        running_loss += loss.item() * bs
        num_samples += bs
        train_bar.set_postfix(loss=f"{running_loss / num_samples:.4f}")

    epoch_loss = running_loss / num_samples if num_samples > 0 else 0.0

    # ── Validation ────────────────────────────────────────────────────────────
    Model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Val   {epoch+1}/{num_epochs}", leave=False)
        for data, target in val_bar:
            data, target = data.to(device, non_blocking=True), target.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                output = Model(data)
            preds = torch.argmax(output, dim=1)
            correct += (preds == target).sum().item()
            total += target.size(0)
            val_bar.set_postfix(acc=f"{correct / total:.4f}")

    val_acc = correct / total if total > 0 else 0.0
    epoch_bar.set_postfix(loss=f"{epoch_loss:.4f}", val_acc=f"{val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_path = os.path.join(save_dir, 'best_model.pth')
        torch.save(Model.state_dict(), best_path)
        tqdm.write(f"Epoch {epoch+1}: saved best model (val_acc={best_val_acc:.4f})")

# Save final model
final_path = os.path.join(save_dir, 'final_model.pth')
torch.save(Model.state_dict(), final_path)
tqdm.write(f"Training finished. Best val acc: {best_val_acc:.4f}. Models saved in {save_dir}")

C:\Users\pstay\AppData\Local\Temp\ipykernel_7068\1213463618.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
Train 1/100:   0%|          | 0/12500 [00:00<?, ?it/s]C:\Users\pstay\AppData\Local\Temp\ipykernel_7068\1213463618.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):

Epochs:   0%|          | 0/100 [02:06<?, ?it/s]                                


KeyboardInterrupt: 